# ch04 练习：缺失值处理

> 共 9 道基础题 + 1 道综合题（拆 3 小步），合计 **12 个空**
> 建议限时：**30 分钟**

数据：`data/device_defects.csv`（270 行 x 13 列）与 `data/load_curve.csv`（6480 行）

**每题都对应竞赛评分表里的一个计分点**。跑通 `assert` 就说明那一步是对的；
跑不通的报错信息本身就告诉你哪里错了。**不要翻答案版。**

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

dev = pd.read_csv("data/device_defects.csv")
print("读入 device_defects.csv ->", dev.shape)

---

### TODO(1) 缺失普查：逐列

1. 逐列缺失计数存入 `miss`（Series，长度 = 列数）
2. 只保留缺失数 > 0 的列，按缺失数**降序**存入 `miss_sorted`

**考点**：`isna().sum()` 给的是**逐列**结果；别忘了 `sort_values`。

In [ ]:
miss = dev.isna().sum()
miss_sorted = miss[miss > 0].sort_values(ascending=False)

# ---- 验收 ----
assert len(miss) == 13, f"miss 应该是逐列的，长度等于列数 13，实际 {len(miss)}"
assert miss_sorted.to_dict() == {"备注": 196, "温度": 23, "负荷值": 21}, (
    f"有缺失的列不对：{miss_sorted.to_dict()}"
)
assert miss_sorted.index[0] == "备注", "缺失最多的一列应该是「备注」"
print("TODO(1) 通过 ->", miss_sorted.to_dict(), "| 合计", int(miss.sum()))

### TODO(2) 缺失率

算出逐列**缺失率（百分数，保留 2 位小数）**，存入 `miss_rate`。

然后取出 `温度` 列的缺失率存入 `temp_miss_rate`（也是百分数，保留 2 位小数）。

In [ ]:
miss_rate = (dev.isna().mean() * 100).round(2)
temp_miss_rate = miss_rate["温度"]

# ---- 验收 ----
assert miss_rate.shape[0] == 13, "miss_rate 应该逐列都有"
assert abs(miss_rate["负荷值"] - 7.78) < 0.01, f"负荷值缺失率应为 7.78%，实际 {miss_rate['负荷值']}"
assert abs(temp_miss_rate - 8.52) < 0.01, f"温度缺失率应为 8.52%，实际 {temp_miss_rate}"
assert abs(miss_rate["备注"] - 72.59) < 0.01, f"备注缺失率应为 72.59%，实际 {miss_rate['备注']}"
print(f"TODO(2) 通过 -> 负荷值 {miss_rate['负荷值']}% | 温度 {temp_miss_rate}% | 备注 {miss_rate['备注']}%")

### TODO(3) 行视角：有多少行受影响

1. 含**任意**缺失的行数存入 `n_rows_any_na`
2. **完全无缺失**的行数存入 `n_rows_complete`
3. **每行**缺失列数存入 `n_na_per_row`（Series，长度 = 行数）

**考点**：`axis=1` 表示"横着看"（沿着列方向压缩）→ 得到**每行**的结果。

In [ ]:
n_rows_any_na = int(dev.isna().any(axis=1).sum())
n_rows_complete = int(dev.notna().all(axis=1).sum())
n_na_per_row = dev.isna().sum(axis=1)

# ---- 验收 ----
assert n_rows_any_na == 211, f"含任意缺失的行应为 211，实际 {n_rows_any_na}"
assert n_rows_complete == 59, f"完全无缺失的行应为 59，实际 {n_rows_complete}"
assert n_rows_any_na + n_rows_complete == len(dev), "两类行数之和应等于总行数"
assert len(n_na_per_row) == 270, "n_na_per_row 是「逐行」的，长度应等于行数"
assert n_na_per_row.max() == 2, f"最多一行缺 2 列，实际最大 {n_na_per_row.max()}"
print(f"TODO(3) 通过 -> 受影响 {n_rows_any_na} 行 / 完整 {n_rows_complete} 行")
print("  每行缺失列数分布:", n_na_per_row.value_counts().sort_index().to_dict())

### TODO(4) `dropna`：默认口径与定点口径

1. 默认 `dropna()`（任缺一列就删）的结果存入 `d_any`
2. 只按 `负荷值` 一列判断的 `dropna` 结果存入 `d_load_only`

**注意对比**：默认口径会把 13 列里**任何**一列缺失的行都删掉，这个数据里会删掉 78% 的行。

In [ ]:
d_any = dev.dropna()
d_load_only = dev.dropna(subset=["负荷值"])

# ---- 验收 ----
assert d_any.shape == (59, 13), f"默认 dropna() 应为 (59, 13)，实际 {d_any.shape}"
assert d_load_only.shape == (249, 13), f"按负荷值删除后应为 (249, 13)，实际 {d_load_only.shape}"
assert d_load_only.shape[0] == len(dev) - miss_sorted["负荷值"], "删除的行数应等于负荷值缺失数"
assert d_load_only["负荷值"].isna().sum() == 0, "定点删除后负荷值不应还有缺失"
print(f"TODO(4) 通过 -> 默认删到 {d_any.shape}，定点删到 {d_load_only.shape}")

### TODO(5) `dropna(thresh=n)`

`thresh=12` 意思是：**这一行至少有 12 个非空值才保留**（表有 13 列，即最多允许缺 1 列）。

把这个结果存入 `d_thresh`，并算出它比原始数据少了多少行，存入 `n_dropped_thresh`。

In [ ]:
d_thresh = dev.dropna(thresh=12)
n_dropped_thresh = len(dev) - len(d_thresh)

# ---- 验收 ----
assert d_thresh.shape == (241, 13), f"thresh=12 应为 (241, 13)，实际 {d_thresh.shape}"
assert n_dropped_thresh == 29, f"应删掉 29 行，实际 {n_dropped_thresh}"
assert n_dropped_thresh == n_na_per_row.eq(2).sum(), (
    "thresh=12 只删「缺 2 列」的行，所以删掉的行数应等于缺 2 列的行数"
)
print(f"TODO(5) 通过 -> thresh=12 剩 {d_thresh.shape[0]} 行，删掉 {n_dropped_thresh} 行")

### TODO(6) `ffill` / `bfill`

对一个有连续缺失的 Series 做前向与后向填充：

1. `load_ff`：`负荷值` 前向填充的结果
2. `load_bf`：`负荷值` 后向填充的结果
3. `n_ff_left`：前向填充后**还剩多少个缺失**
4. `n_bf_left`：后向填充后**还剩多少个缺失**

**考点**：pandas 3.0 起 `fillna(method="ffill")` 已移除，要用 `.ffill()` / `.bfill()`。

In [ ]:
load_ff = dev["负荷值"].ffill()
load_bf = dev["负荷值"].bfill()
n_ff_left = int(load_ff.isna().sum())
n_bf_left = int(load_bf.isna().sum())

# ---- 验收 ----
assert n_ff_left == 0, f"前向填完后不应还有缺失，实际 {n_ff_left}"
assert n_bf_left == 0, f"后向填完后不应还有缺失，实际 {n_bf_left}"
assert load_ff.iloc[0] == dev["负荷值"].iloc[0] or pd.isna(dev["负荷值"].iloc[0]), (
    "第一行前面没有值，前向填充不会改动第一行"
)
assert len(load_ff) == len(dev), "填充不改变行数"
print(f"TODO(6) 通过 -> ffill 残余 {n_ff_left} 个，bfill 残余 {n_bf_left} 个")
print("  注意：ffill 会把连续缺失段全部填成同一个值，长缺失段会被压成水平线")

### TODO(7) 统计量填充：中位数

1. 计算出 `负荷值` 的**中位数**，存入 `load_median`
2. 用这个中位数填充缺失，结果存入 `load_fill_median`
3. 填入的值本身存入 `n_filled_median`（应该等于缺失数）

**注意**：这里必须先清异常值才谈得上"正确的均值"。本数据掺了 9999 哨兵值，
所以均值 379.68 完全不可用，**中位数 87.34 才是可用统计量**。

In [ ]:
load_median = dev["负荷值"].median()
load_fill_median = dev["负荷值"].fillna(load_median)
n_filled_median = int(dev["负荷值"].isna().sum() - load_fill_median.isna().sum())

# ---- 验收 ----
assert abs(load_median - 87.34) < 1e-6, f"负荷值中位数应为 87.34，实际 {load_median}"
assert load_fill_median.isna().sum() == 0, "填充后不应还有缺失"
assert n_filled_median == 21, f"应填入 21 个值，实际 {n_filled_median}"
assert abs(load_fill_median.mean() - 356.9467) < 1e-3, (
    f"中位数填充后均值应约 356.95，实际 {load_fill_median.mean():.4f}"
)
print(f"TODO(7) 通过 -> median={load_median}, 填后 mean={load_fill_median.mean():.4f}")

### TODO(8) 分组填充（MAR 的正确姿势）

按 `设备类型` 分组，用**组内中位数**填充 `负荷值`，结果存入 `load_fill_group`。

**这是本章最重要的一个坑**：不能写 `load.fillna(med_by_type)`——
那个 Series 的索引是"设备类型"，与行索引对不上，会**静默地一个都填不上**。

正确写法二选一：
- `dev.groupby("设备类型")["负荷值"].transform("median")`
- `dev["设备类型"].map(med_by_type)`

In [ ]:
med_by_type = dev.groupby("设备类型")["负荷值"].median()

load_fill_group = dev["负荷值"].fillna(
    dev.groupby("设备类型")["负荷值"].transform("median")
)

# ---- 验收 ----
assert load_fill_group.isna().sum() == 0, "分组填充后不应还有缺失"
mask_tr = dev["负荷值"].isna() & (dev["设备类型"] == "变压器")
assert mask_tr.sum() == 6, f"变压器组原本应缺 6 行，实际 {mask_tr.sum()}"
assert np.allclose(load_fill_group[mask_tr].unique(), [521.08]), (
    f"变压器组缺失行应被填成组内中位数 521.08，实际 {load_fill_group[mask_tr].unique()}"
)
print(f"TODO(8) 通过 -> 分组填充后 mean={load_fill_group.mean():.4f}")
print(f"  对比：全局中位数填 {load_fill_median.mean():.4f} <- 抹平了组间差异")

### TODO(9) 时序插值（`load_curve.csv`）

按 `台区编号` 分组，对每组的 `负荷值` 做**线性插值**，结果存入 `curve_filled`（DataFrame）。

然后再算 A 台区（`STATION_A_01`）插值后的 `负荷值` 均值，存入 `a01_filled_mean`。

**为什么必须分组插值**：曲线是 3 个台区**首尾拼接**的，
不分组的话 A 台区的最后一段会和 C 台区的开头连起来插，插出跨台区的假数据。

In [ ]:
curve = pd.read_csv("data/load_curve.csv", parse_dates=["时间戳"])
print("load_curve.csv ->", curve.shape, "| 负荷值缺失:", int(curve["负荷值"].isna().sum()))
print("按台区缺失:", curve.groupby("台区编号")["负荷值"].apply(lambda s: int(s.isna().sum())).to_dict())


def _interp(s: pd.Series) -> pd.Series:
    """对一组做线性插值（交互式定义，避免在挖空块里写 lambda 换行）。"""
    return s.interpolate(method="linear")


curve_filled = curve.copy()
curve_filled["负荷值"] = curve_filled.groupby("台区编号")["负荷值"].transform(_interp)
a01_filled_mean = curve_filled.loc[curve_filled["台区编号"] == "STATION_A_01", "负荷值"].mean()

# ---- 验收 ----
assert curve_filled.shape == (6480, 5), f"形状不应改变，实际 {curve_filled.shape}"
assert curve_filled["负荷值"].isna().sum() == 0, "分组插值后不应还有缺失"
assert abs(a01_filled_mean - 660.22) < 0.05, f"A 台区插值后均值应约 660.22，实际 {a01_filled_mean:.2f}"
assert len(curve_filled) == len(curve), "行数不应改变"
print(f"TODO(9) 通过 -> 插值后总缺失 {curve_filled['负荷值'].isna().sum()}，A 台区均值 {a01_filled_mean:.2f}")

---

## 综合题：从「判定机制」到「清洗报告」

| 步骤 | 产物 | 对应评分点 |
|---|---|---|
| ① 判定机制（MAR 实证） | `rate_cd` / `rate_other` | 数据清洗 · 策略选择依据 |
| ② 缺失指示列 | `dev2` 带 `温度_缺失` 列 | 数据清洗 · 特征构造 |
| ③ 组装清洗报告 | `report` | 数据探索 · 结论输出 |

#### 综合 ① 判定 `温度` 缺失的机制

把台区分成两组：`STATION_C_*` / `STATION_D_*` 为 `True`，其余 `False`。

1. 两组的温度缺失率（**百分数，保留 2 位**）分别存入 `rate_cd`、`rate_other`
2. 给出机制判定字符串存入 `temp_mechanism`，取值必须是 `"MCAR"` / `"MAR"` / `"MNAR"` 之一

判定规则：若两组缺失率**差距巨大**（此处 C/D 组 ≈ 23.7%、其他组 = 0%），
说明缺失与「台区」这个已观测变量强烈相关 → **MAR**。

In [ ]:
is_cd = dev["台区编号"].str.startswith(("STATION_C_", "STATION_D_"))
print("C/D 台区行数:", int(is_cd.sum()), "| 其他台区行数:", int((~is_cd).sum()))

rate_cd = round(dev.loc[is_cd, "温度"].isna().mean() * 100, 2)
rate_other = round(dev.loc[~is_cd, "温度"].isna().mean() * 100, 2)
temp_mechanism = "MAR"

# ---- 验收 ----
assert abs(rate_cd - 23.71) < 0.02, f"C/D 组缺失率应约 23.71%，实际 {rate_cd}"
assert rate_other == 0.0, f"其他台区不应有温度缺失，实际 {rate_other}"
assert temp_mechanism in {"MCAR", "MAR", "MNAR"}, "机制取值必须是 MCAR / MAR / MNAR 之一"
assert temp_mechanism == "MAR", "缺失与已观测变量（台区）强相关 -> MAR"
print(f"综合 ① 通过 -> C/D 组 {rate_cd}% vs 其他组 {rate_other}% -> {temp_mechanism}")

#### 综合 ② 缺失指示列

在 `dev` 的副本 `dev2` 上新增一列 `温度_缺失`，取值 0 / 1（整数）。

然后算出该列的求和，存入 `n_temp_flag`。

In [ ]:
dev2 = dev.copy()
dev2["温度_缺失"] = dev2["温度"].isna().astype(int)
n_temp_flag = int(dev2["温度_缺失"].sum())

# ---- 验收 ----
assert "温度_缺失" in dev2.columns, "dev2 应该有「温度_缺失」列"
assert "温度_缺失" not in dev.columns, "不应该直接改原 dev（要用副本）"
assert dev2["温度_缺失"].dtype == np.int64 or str(dev2["温度_缺失"].dtype).startswith("int"), (
    f"指示列应该是整数型，实际 {dev2['温度_缺失'].dtype}"
)
assert n_temp_flag == 23, f"指示列之和应等于温度缺失数 23，实际 {n_temp_flag}"
assert set(dev2["温度_缺失"].unique()) <= {0, 1}, "指示列只能是 0 或 1"
print(f"综合 ② 通过 -> 指示列合计 {n_temp_flag}，取值集合 {sorted(dev2['温度_缺失'].unique())}")

#### 综合 ③ 组装清洗报告

把这一章算出来的结论组装成 `report` 字典，键必须完全一致：

```
n_cols / n_rows / n_missing_cells / n_rows_any_na / n_rows_complete
worst_col / worst_col_missing / temp_mechanism
load_median / load_mean_after_fill
```

其中：
- `worst_col`：缺失最多的列名；`worst_col_missing`：它的缺失数
- `load_median`：`负荷值` 中位数
- `load_mean_after_fill`：用中位数填充后的整体均值
- 其余引用前面已算好的变量

In [ ]:
report = {
    "n_cols": dev.shape[1],
    "n_rows": dev.shape[0],
    "n_missing_cells": int(miss.sum()),
    "n_rows_any_na": n_rows_any_na,
    "n_rows_complete": n_rows_complete,
    "worst_col": miss.idxmax(),
    "worst_col_missing": int(miss.max()),
    "temp_mechanism": temp_mechanism,
    "load_median": load_median,
    "load_mean_after_fill": round(float(load_fill_median.mean()), 4),
}

# ---- 验收 ----
expect_keys = {
    "n_cols", "n_rows", "n_missing_cells", "n_rows_any_na", "n_rows_complete",
    "worst_col", "worst_col_missing", "temp_mechanism",
    "load_median", "load_mean_after_fill",
}
assert isinstance(report, dict), "report 应该是字典"
assert set(report) == expect_keys, (
    f"键不对。多的是 {set(report) - expect_keys}，缺的是 {expect_keys - set(report)}"
)
assert report["n_cols"] == 13 and report["n_rows"] == 270
assert report["n_missing_cells"] == 240, "全表缺失总量应为 240"
assert report["n_rows_any_na"] == 211 and report["n_rows_complete"] == 59
assert report["worst_col"] == "备注" and report["worst_col_missing"] == 196
assert report["temp_mechanism"] == "MAR"
assert abs(report["load_median"] - 87.34) < 1e-6
assert abs(report["load_mean_after_fill"] - 356.9467) < 1e-3
assert report["n_rows_any_na"] + report["n_rows_complete"] == report["n_rows"], "行数应自洽"
print("综合 ③ 通过 -> 清洗报告：")
for k, v in report.items():
    print(f"  {k:22s} = {v!r}")

---

### 复盘提问

**复盘问自己**（三题都能答上来才算过）：

1. `fillna(dev.groupby("设备类型")["负荷值"].median())` 为什么一个都填不上？怎么改？
2. 为什么这个数据的 `负荷值` 要用中位数填而不是均值？均值是多少、中位数是多少？
3. 时序连续缺失段，`ffill` 和 `interpolate` 的结果差在哪？为什么后者更适合负荷曲线？

答不上来的，回 `_notes/错题本.md` 记一笔。